In [ ]:
%%capture
!pip install scikit-learn matplotlib pymupdf langgraph langchain langsmith python-dotenv langchain-groq langchain_tavily langchain-community langchain-text-splitters faiss-cpu

In [ ]:
%%capture
!pip install -U "transformers==5.23.0" "tokenizers==0.23.1" datasets

In [ ]:
import pymupdf  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langchain_core.embeddings import Embeddings
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [ ]:
###Clip Model
import os
from dotenv import load_dotenv
load_dotenv()

### initialize the Clip Model for unified embeddings
clip_model=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor=CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

In [ ]:
### Embedding functions
def _normalize_clip_output(output):
    """Return a normalized 1D numpy vector from CLIP output."""
    if isinstance(output, torch.Tensor):
        features = output
    elif hasattr(output, "image_embeds") and output.image_embeds is not None:
        features = output.image_embeds
    elif hasattr(output, "text_embeds") and output.text_embeds is not None:
        features = output.text_embeds
    elif hasattr(output, "pooler_output") and output.pooler_output is not None:
        features = output.pooler_output
    elif hasattr(output, "last_hidden_state") and output.last_hidden_state is not None:
        features = output.last_hidden_state[:, 0, :]
    else:
        raise ValueError("Unsupported CLIP output format")

    features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().detach().cpu().numpy()


def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data, str):  # If path
        image = Image.open(image_data).convert("RGB")
    else:  # If PIL Image
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        output = clip_model.get_image_features(**inputs)
        return _normalize_clip_output(output)


def embed_text(text):
    """Embed text using CLIP."""
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77  # CLIP's max token length
    )
    with torch.no_grad():
        output = clip_model.get_text_features(**inputs)
        return _normalize_clip_output(output)


class CLIPTextEmbeddings(Embeddings):
    """LangChain Embeddings adapter backed by CLIP text encoder."""

    def embed_documents(self, texts):
        return [embed_text(text).tolist() for text in texts]

    def embed_query(self, text):
        return embed_text(text).tolist()


embedding_model = CLIPTextEmbeddings()

In [ ]:
# ## Process PDF
# pdf_path="pathologyBooks/harsh-mohan-textbook-of-pathology-6th-ed.pdf"
# doc=pymupdf.open(pdf_path)
# # Storage for all documents and embeddings
# all_docs = []
# all_embeddings = []
# image_data_store = {}  # Store actual image data for LLM

# # Text splitter
# splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
from datasets import load_dataset

ds = load_dataset("flaviagiammarino/path-vqa")

In [ ]:
# for i,page in enumerate(doc):
#     ## process text
#     text=page.get_text()
#     if text.strip():
#         ##create temporary document for splitting
#         temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
#         text_chunks = splitter.split_documents([temp_doc])

#         #Embed each chunk using CLIP
#         for chunk in text_chunks:
#             embedding = embed_text(chunk.page_content)
#             all_embeddings.append(embedding)
#             all_docs.append(chunk)



#     ## process images
#     ##Three Important Actions:

#     ##Convert PDF image to PIL format
#     ##Store as base64 for GPT-4V (which needs base64 images)
#     ##Create CLIP embedding for retrieval

#     for img_index, img in enumerate(page.get_images(full=True)):
#         try:
#             xref = img[0]
#             base_image = doc.extract_image(xref)
#             image_bytes = base_image["image"]
            
#             # Convert to PIL Image
#             pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
#             # Create unique identifier
#             image_id = f"page_{i}_img_{img_index}"
            
#             # Store image as base64 for later use with GPT-4V
#             buffered = io.BytesIO()
#             pil_image.save(buffered, format="PNG")
#             img_base64 = base64.b64encode(buffered.getvalue()).decode()
#             image_data_store[image_id] = img_base64
            
#             # Embed image using CLIP
#             embedding = embed_image(pil_image)
#             all_embeddings.append(embedding)
            
#             # Create document for image
#             image_doc = Document(
#                 page_content=f"[Image: {image_id}]",
#                 metadata={"page": i, "type": "image", "image_id": image_id}
#             )
#             all_docs.append(image_doc)
            
#         except Exception as e:
#             print(f"Error processing image {img_index} on page {i}: {e}")
#             continue

# doc.close()

In [ ]:
# Build multimodal docs from PathVQA train split (instead of PDF pages)
train_ds = ds["train"]

# Set to None to process the full train split
max_samples = 1000

all_docs = []
all_embeddings = []
image_data_store = {}

if max_samples is not None:
    sample_count = min(max_samples, len(train_ds))
    train_ds = train_ds.select(range(sample_count))

for i, sample in enumerate(train_ds):
    try:
        image = sample["image"]
        pil_image = image.convert("RGB") if hasattr(image, "convert") else Image.open(image).convert("RGB")

        question = str(sample.get("question", "")).strip()
        answer_value = sample.get("answer", sample.get("answers", ""))
        if isinstance(answer_value, (list, tuple)):
            answer = " | ".join(str(a) for a in answer_value)
        else:
            answer = str(answer_value).strip()

        image_id = f"train_{i}"
        buffered = io.BytesIO()
        pil_image.save(buffered, format="PNG")
        image_data_store[image_id] = base64.b64encode(buffered.getvalue()).decode()

        embedding = embed_image(pil_image)
        all_embeddings.append(embedding)

        image_doc = Document(
            page_content=f"[Image: {image_id}]",
            metadata={
                "source": "path-vqa",
                "split": "train",
                "index": i,
                "type": "image",
                "image_id": image_id,
                "question": question,
                "answer": answer
            }
        )
        all_docs.append(image_doc)

    except Exception as e:
        print(f"Error processing train sample {i}: {e}")
        continue

len(all_docs), len(all_embeddings)

In [ ]:
# Create unified FAISS vector store with CLIP embeddings
embeddings_array = np.array(all_embeddings)
embeddings_array

In [ ]:
# (all_docs,embeddings_array)

In [ ]:
# Create custom FAISS index since we have precomputed embeddings
paired_docs_embeddings = list(zip(all_docs, embeddings_array))
text_embeddings = [(doc.page_content, emb) for doc, emb in paired_docs_embeddings]
metadatas = [doc.metadata for doc, _ in paired_docs_embeddings]

vector_store = FAISS.from_embeddings(
    text_embeddings=text_embeddings,
    embedding=embedding_model,
    metadatas=metadatas
)
vector_store

In [ ]:
def retrieve_multimodal(query, k=5, mode="text"):
    """Unified retrieval using CLIP embeddings for text or image queries."""
    if mode == "image":
        query_embedding = embed_image(query)
    elif mode == "text":
        query_embedding = embed_text(query)
    else:
        raise ValueError("mode must be either 'text' or 'image'")

    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )

    return results

In [ ]:
import matplotlib.pyplot as plt

# Pick one sample from the test split
test_sample_idx = 0
test_sample = ds["test"][test_sample_idx]
test_image = test_sample["image"]
test_question = str(test_sample.get("question", "")).strip()
test_answer = str(test_sample.get("answer", test_sample.get("answers", ""))).strip()

print(f"Test sample index: {test_sample_idx}")
print(f"Question: {test_question}")
print(f"Ground-truth answer: {test_answer}")

# Use the test image as query for image-to-image retrieval
retrieved_docs = retrieve_multimodal(test_image, k=5, mode="image")

# Show query image
plt.figure(figsize=(4, 4))
plt.imshow(test_image)
plt.axis("off")
plt.title("Query image (from test split)")
plt.show()

# Show retrieved similar images
image_results = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]

if not image_results:
    print("No image documents were retrieved.")
else:
    cols = min(3, len(image_results))
    rows = (len(image_results) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows == 1 or cols == 1:
        axes = list(axes)
    else:
        axes = [ax for row_axes in axes for ax in row_axes]

    for idx, doc in enumerate(image_results):
        ax = axes[idx]
        image_id = doc.metadata.get("image_id")
        b64_data = image_data_store.get(image_id)
        if b64_data:
            img = Image.open(io.BytesIO(base64.b64decode(b64_data))).convert("RGB")
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "Image not found", ha="center", va="center")
        ax.axis("off")
        pred_q = doc.metadata.get("question", "")
        pred_a = doc.metadata.get("answer", "")
        ax.set_title(f"Top {idx+1}\nQ: {pred_q[:40]}\nA: {pred_a[:40]}", fontsize=9)

    for j in range(len(image_results), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

retrieved_docs

In [ ]:
# !pip install -U "huggingface_hub" --quiet

In [ ]:
# # Validate Hugging Face token (no interactive login required)
# from huggingface_hub import HfApi

# HF_TOKEN =
# if not HF_TOKEN:
#     raise ValueError("HF_TOKEN is not set. Add it to your environment or set token variable in notebook.")

# api = HfApi(token=HF_TOKEN)
# user_info = api.whoami()
# print(f"Hugging Face auth OK for: {user_info.get('name', 'unknown')}")

In [ ]:
# from huggingface_hub import snapshot_download

# model_id = "google/medgemma-4b-it"
# local_dir = "../google/medgemma-4b-it"


# download_path = snapshot_download(
#     repo_id=model_id,
#     local_dir=local_dir,
#     ignore_patterns=["original/*"],
#     token=HF_TOKEN
# )

# print(f"Snapshot resolved: {download_path}")

In [ ]:
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="../google/medgemma-4b-it")

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages)

In [ ]:
pipe(text=messages)[0]['generated_text'][1]['content']

In [ ]:
def create_multimodal_message(query, retrieved_docs):
    """Create a message with both text and images for GPT-4V."""
    content = []
    
    # Add the query
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    # Separate text and image documents
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    # Add text context
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    # Add images
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    # Add instruction
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [ ]:
def multimodal_pdf_rag_pipeline(query):
    """Main pipeline for multimodal RAG."""
    # Retrieve relevant documents
    context_docs = retrieve_multimodal(query, k=5)
    
    # Create multimodal message
    message = create_multimodal_message(query, context_docs)
    
    # Get response from GPT-4V
    response = pipe(message.content)
    
    # Print retrieved context info
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    return response

In [ ]:
queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
for query in queries:
    print(f"\nQuery: {query}")
    print("-" * 50)
    answer = multimodal_pdf_rag_pipeline(query)
    print(f"Answer: {answer}")
    print("=" * 70)